# Colab RGB SRNet — prospective 45-minute cumulative amendment

CPU-default metadata preparation; all fitting/export switches are False.
The consumed T4 profile is reused through pinned receipt/audit/source metadata.
This notebook has no profile execution cell and generates no fitting permission.
All previous session charges remain; 300 seconds of future setup/export are reserved.
At most 150 seconds precede launch; another 150 remain for export/runtime deletion.
Before a later approved GPU launch, the operator must supply the observed allocation
age in the first cell; a missing age or exhausted reservation prevents fitting.
Current fixed recipe: two seeds /20 epochs /640 updates, 64 train /20 development.
Reserved 21 RGB and all 13 L pixels remain closed; actual roles stay unassigned.
No hard cgroup RAM/zero-swap or whole-driver VRAM guarantee is claimed.
The new allowance and exact fit contract require later explicit user approval.
On completion or interruption, preserve evidence and delete the GPU runtime.


In [ ]:
import hashlib
import json
import os
import re
import sys
import time
from pathlib import Path

from google.colab import files

setup_started = time.monotonic()
ALLOCATION_REQUEST_AGE_AT_BOOTSTRAP_SECONDS = None

SOURCE_SHA256 = 'a465948b02d85321f802535143ddccb0a0d781f7bb8f029b8b99427013b951f1'
if any(n == "core" or n.startswith("core.") for n in sys.modules):
    raise RuntimeError("Use a fresh kernel; preserve every consumed attempt")
uploaded = files.upload()  # Select the accompanying sources.json only.
if set(uploaded) != {'png_bmp_colab_amended_fit_sources_20261107.json'}:
    raise ValueError("Select sources.json only")
raw = uploaded['png_bmp_colab_amended_fit_sources_20261107.json']
if len(raw) > 4 * 1024**2 or hashlib.sha256(raw).hexdigest() != SOURCE_SHA256:
    raise ValueError("Reviewed source packet hash/extent mismatch")
packet = json.loads(raw)
if (set(packet) != {"schema", "base_commit", "files"}
        or packet["schema"] != "colab-native-source-package-v1"
        or packet["base_commit"] != '2f41bfe2e95febcaab80f80ab4875c72073c3a32'
        or not 3 <= len(packet["files"]) <= 513 or "LICENSE" not in packet["files"]):
    raise ValueError("Reviewed source packet schema mismatch")
prepared = {}
for name, row in packet["files"].items():
    allowed = re.fullmatch(
        r"(?:core|steganography)/(?:[a-zA-Z0-9_]+/){0,3}[a-zA-Z0-9_]+\.py", name
    )
    if name != "LICENSE" and allowed is None:
        raise ValueError("Reviewed source path mismatch")
    if set(row) != {"utf8", "sha256"} or not isinstance(row["utf8"], str):
        raise ValueError("Reviewed source entry mismatch")
    content = row["utf8"].encode()
    if len(content) > 1024**2 or hashlib.sha256(content).hexdigest() != row["sha256"]:
        raise ValueError("Reviewed source file hash/extent mismatch")
    prepared[name] = content
source_root = Path("/content/spatial-colab-amended-fit-source-20261107")
if any(p.is_symlink() for p in (source_root, *source_root.parents)):
    raise ValueError("Source directory symlink")
source_root.mkdir(mode=0o700)
for name, content in prepared.items():
    target = source_root / name
    target.parent.mkdir(mode=0o700, parents=True, exist_ok=True)
    fd = os.open(target, os.O_WRONLY | os.O_CREAT | os.O_EXCL | os.O_NOFOLLOW, 0o600)
    with os.fdopen(fd, "wb") as stream:
        stream.write(content)
sys.path.insert(0, str(source_root))
print("Reviewed source package installed:", SOURCE_SHA256)


In [ ]:
import base64

from core import colab_artifacts as artifacts
from core import colab_execution as controller
from core import spatial_colab_budget as budget
from core import spatial_colab_fit as training
from core import spatial_colab_handoff as handoff
from core import spatial_colab_profile as profile

metadata_raw = base64.b64decode(
    '...'
)
METADATA_SHA256 = '58d314622b77d7f79c674c3b947ea47895ab04693380a06d51a44b9ac548465e'
if artifacts.digest(metadata_raw) != METADATA_SHA256:
    raise ValueError("Immutable profile metadata changed")
profile_output = source_root / "accepted-profile"
artifacts.directory(profile_output)
artifacts.put(profile_output / "profile-metadata.json", metadata_raw, budget.PACKET_LIMIT)
fit_protocol = source_root / "fit-contract.json"
debit = {"schema": "colab-fit-external-debit-v1", "complete": True,
         "seconds": 1250, "output_bytes": 71303168}
training.freeze(source_root, profile_output, debit, fit_protocol, amended=True)
fit_job, fit_binding, fit_limits = controller.source_contract(source_root, fit_protocol)
if fit_binding != '7a45cab6774388e9eb156af90eb4c10c962f522a6a2ad6678853fb88e65d7b94':
    raise ValueError("Reviewable fit contract changed")
fit_permission_verified = False
print("Fit proposal:", fit_binding)
print("Cumulative seconds:", budget.SECONDS, "remaining:", fit_limits.seconds)
print("No profile rerun; separate fitting permission missing")


In [ ]:
VERIFY_APPROVED_FIT = False
if VERIFY_APPROVED_FIT:
    selected = files.upload()  # Later approval of this exact v2 contract.
    if set(selected) != {"fit-permission.json"}:
        raise ValueError("Select fit-permission.json only")
    fit_permission = source_root / "fit-permission.json"
    artifacts.put(fit_permission, selected["fit-permission.json"])
    controller.permission(fit_job, fit_binding, fit_permission)
    fit_permission_verified = True


In [ ]:
RUN_APPROVED_FIT = False
TRAIN_PACKAGE_SHA256 = "1773c2991478e40718ac6de37e6705141e7a795d02a8fb80b029dc8173d58327"
DEVELOPMENT_PACKAGE_SHA256 = ""  # Prepare only after later fitting approval.
if RUN_APPROVED_FIT:
    if not fit_permission_verified:
        raise ValueError("Separate fitting permission required before media upload")
    budget.setup_age(ALLOCATION_REQUEST_AGE_AT_BOOTSTRAP_SECONDS,
                     time.monotonic() - setup_started)
    artifacts.identity(DEVELOPMENT_PACKAGE_SHA256)
    selected = files.upload()
    if set(selected) != {"train-cache.json", "development-cache.json"}:
        raise ValueError("Select only the approved train/development packets")
    split = profile.engine.split_document(fit_job["payload"]["split"])
    train_packet = source_root / "train-cache.json"
    development_packet = source_root / "development-cache.json"
    artifacts.put(train_packet, selected["train-cache.json"], handoff.CACHE_LIMIT)
    artifacts.put(development_packet, selected["development-cache.json"], handoff.CACHE_LIMIT)
    cache = handoff.install_cache(train_packet, TRAIN_PACKAGE_SHA256, split, source_root / "cache")
    handoff.install_development(cache, split, development_packet, DEVELOPMENT_PACKAGE_SHA256)
    budget.setup_age(ALLOCATION_REQUEST_AGE_AT_BOOTSTRAP_SECONDS,
                     time.monotonic() - setup_started)
    fit_output = source_root / "fit-output"
    fit_result = controller.run(source_root, fit_protocol, fit_output, cache=cache.root,
                                permission_path=fit_permission, profile_out=profile_output)
    controller.audit(source_root, fit_protocol, fit_output)
    print("Execution:", fit_result["status"])


In [ ]:
DOWNLOAD_VERIFIED_TRAINING = False
if DOWNLOAD_VERIFIED_TRAINING:
    checked = controller.audit(source_root, fit_protocol, fit_output)
    if checked["status"] != "completed":
        raise ValueError("Keep unavailable evidence private; do not claim a complete fit")
    archive = fit_output / "verified-training.zip"
    checksum = checked["output_files"][archive.name]["sha256"]
    handoff.verify_evidence(archive, checksum, fit_binding, max_members=4097)
    files.download(str(archive))
    for cell in checked["worker"]["cells"]:
        for checkpoint in cell["checkpoints"]:
            relative = "work/fit/" + checkpoint["name"]
            path = fit_output / relative
            expected_sha = checked["output_files"][relative]["sha256"]
            if handoff.stream_hash(path, fit_limits.file_bytes) != expected_sha:
                raise ValueError("Checkpoint changed before export")
            files.download(str(path))
    files.download(str(fit_output / "closed.json"))
